# 🔄 LNG Deal Lifecycle — Execution to Settlement (Option B)
### Loading · In-Transit · Discharge · Invoice · GL Entries · Settlement · Month-End

This notebook continues from [`lng_feasibility_extended.ipynb`](lng_feasibility_extended.ipynb) (Option A).

**Phases covered:**
| Phase | What Happens |
|---|---|
| 1 | Load confirmed deal JSON |
| 2 | Bill of Lading — volume confirmation & title transfer |
| 3 | In-Transit lifecycle — daily BOG accruals, cross-month detection |
| 4 | Discharge & Outturn — final volume, quality |
| 5 | Demurrage calculator — laytime, NOR, COSP |
| 6 | Invoice generation — outturn volume × final price |
| 7 | GL Journal entries — complete DR/CR at every stage |
| 8 | Settlement — AR/AP clearance, netting, cash |
| 9 | Month-end close checklist |
| 10 | Final P&L reconciliation |

---
> 💡 **To link notebooks:** Run Section 7 in `lng_feasibility_extended.ipynb` to export `lng_deal_extended_YYYYMMDD.json`, then load it in Cell 2 below. Or load from `lng_feasibility.ipynb` directly.
---

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — ENVIRONMENT SETUP (Colab + Local Jupyter)
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys, shutil

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

for pkg in ['ipywidgets', 'scipy']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

if not IN_COLAB:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'widgetsnbextension', 'jupyterlab-widgets'])
        if shutil.which('jupyter'):
            subprocess.run(['jupyter', 'nbextension', 'enable', '--py',
                            'widgetsnbextension', '--sys-prefix'],
                           capture_output=True)
    except Exception:
        pass

print('✅  Environment ready.', '(Colab)' if IN_COLAB else '(Local Jupyter)')
if not IN_COLAB:
    print('   ℹ️  If sliders don\'t appear, restart the notebook server after this first run.')

In [ ]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import date, timedelta, datetime
import json, os, glob

%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',      'ytick.color': '#8b949e',
    'text.color':  '#e6edf3',      'grid.color':  '#21262d',
    'grid.linestyle': '--',        'grid.alpha':  0.5,
    'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold', 'figure.dpi': 110,
})
C = {
    'gain':'#3fb950','loss':'#f85149','warn':'#d29922',
    'blue':'#58a6ff','purple':'#bc8cff','teal':'#39d353',
    'bg':'#0d1117','card':'#161b22','border':'#30363d',
    'text':'#e6edf3','muted':'#8b949e','dr':'#58a6ff','cr':'#d29922',
}
style  = {'description_width': '220px'}
layout = widgets.Layout(width='520px')

# Global journal ledger
JOURNAL = []

def add_journal(event, account, dr_cr, amount, gl_code, narrative=''):
    JOURNAL.append({
        'Event':     event,
        'Account':   account,
        'DR/CR':     dr_cr,
        'Amount':    round(amount, 2),
        'GL Code':   gl_code,
        'Narrative': narrative,
        'Date':      str(date.today()),
    })

def show_journal(filter_event=None):
    if not JOURNAL:
        print("No journal entries yet.")
        return
    df = pd.DataFrame(JOURNAL)
    if filter_event:
        df = df[df['Event'] == filter_event]
    df['Amount'] = df['Amount'].map('${:,.2f}'.format)
    display(df[['Event','Account','DR/CR','Amount','GL Code','Narrative']])
    totals = pd.DataFrame(JOURNAL)
    dr = totals[totals['DR/CR']=='DR']['Amount'].sum()
    cr = totals[totals['DR/CR']=='CR']['Amount'].sum()
    balanced = abs(dr - cr) < 0.01
    print(f"  Total DR: ${dr:,.2f}  |  Total CR: ${cr:,.2f}  |  {'✅ BALANCED' if balanced else '❌ OUT OF BALANCE'}")

print("✅  Option B loaded. Proceed to Cell 2 to load your deal.")


---
## 📂 Phase 1 — Load Confirmed Deal
*Import from Option A extended JSON, Option A basic JSON, or enter manually.*

In [ ]:
# ── CELL 2: Load deal ─────────────────────────────────────────────────────────
def refresh_json_list():
    found = sorted(glob.glob('lng_deal_*.json'), reverse=True)
    return found if found else ['No JSON found']

all_jsons = refresh_json_list()

w_json = widgets.Dropdown(
    options=all_jsons, value=all_jsons[0] if all_jsons else '',
    description='Load JSON:', style=style, layout=layout
)
w_load_btn2 = widgets.Button(
    description='📂  Load Deal',
    button_style='info',
    layout=widgets.Layout(width='160px', height='36px')
)

# Deal state
DS = {
    'net_mmbtu':   1362090.0,
    'buy_px':      13.28,
    'sell_px':     14.20,
    'hedge_ratio': 0.80,
    'hedge_px':    13.75,
    'oci_balance': 0.0,
    'cpty':        'UK Gas Trading Ltd',
    'buy_cpty':    'QatarEnergy',
    'book':        'ASIA_ST_SPOT',
    'trader':      'Sarah Chen',
}

# Manual overrides
w_net_mmbtu2   = widgets.FloatText(value=1362090, description='Net MMBtu (gross−BOG est.):', style=style, layout=layout)
w_buy_px2      = widgets.FloatText(value=13.28, description='Buy Price ($/MMBtu):', style=style, layout=layout)
w_sell_px2     = widgets.FloatText(value=14.20, description='Sell Price ($/MMBtu):', style=style, layout=layout)
w_cpty2        = widgets.Text(value='UK Gas Trading Ltd', description='Sell Counterparty:', style=style, layout=layout)
w_buy_cpty2    = widgets.Text(value='QatarEnergy', description='Buy Counterparty:', style=style, layout=layout)
w_total_voyage = widgets.FloatText(value=4726258, description='Total Voyage Costs ($):', style=style, layout=layout)
w_regas2       = widgets.FloatText(value=544836, description='Regasification Costs ($):', style=style, layout=layout)
w_bog_rate2    = widgets.FloatSlider(value=0.10, min=0.05, max=0.20, step=0.005,
    description='BOG Rate (%/day):', style=style, layout=layout, readout_format='.3f')
w_voyage_days2 = widgets.IntSlider(value=20, min=5, max=35,
    description='Total Voyage Days:', style=style, layout=layout)

load_out2 = widgets.Output()

def load_deal2(_=None):
    global DS
    fname = w_json.value
    with load_out2:
        clear_output(wait=True)
        if fname and os.path.exists(fname):
            with open(fname) as f:
                data = json.load(f)
            # Handle both extended and base format
            pricing = data.get('pricing', data.get('pnl', {}))
            vols    = data.get('volumes', {})
            pnl_    = data.get('pnl_ifrs9', data.get('pnl', {}))
            DS.update({
                'net_mmbtu':   float(vols.get('net_mmbtu',
                               vols.get('net_delivered_mmbtu', 1362090))),
                'buy_px':      float(pricing.get('buy_px',
                               pricing.get('buy_price_mmbtu', 13.28))),
                'sell_px':     float(pricing.get('sell_px',
                               pricing.get('sell_price_mmbtu', 14.20))),
                'hedge_ratio': float(pricing.get('hedge_ratio', 0.80)),
                'hedge_px':    float(pricing.get('hedge_px', 13.75)),
                'cpty':        data.get('counterparty',
                               DS['cpty']),
                'trader':      data.get('trader', DS['trader']),
                'book':        data.get('book', DS['book']),
            })
            w_net_mmbtu2.value = DS['net_mmbtu']
            w_buy_px2.value    = DS['buy_px']
            w_sell_px2.value   = DS['sell_px']
            w_cpty2.value      = DS['cpty']
            print(f"✅  Loaded: {fname}")
            print(f"   Net MMBtu:  {DS['net_mmbtu']:,.0f}")
            print(f"   Buy Price:  ${DS['buy_px']:.4f}")
            print(f"   Sell Price: ${DS['sell_px']:.4f}")
            print(f"   Hedge Ratio:{DS['hedge_ratio']*100:.0f}%")
        else:
            print(f"⚠️  {fname} not found — using manual inputs.")

w_load_btn2.on_click(load_deal2)

# ── Upload support (Colab + Local Jupyter) ────────────────────────────────────
upload_out2 = widgets.Output()

if IN_COLAB:
    w_upload_btn2 = widgets.Button(
        description='📤  Upload JSON file',
        button_style='warning',
        layout=widgets.Layout(width='200px', height='36px')
    )
    def _colab_upload2(_=None):
        with upload_out2:
            clear_output(wait=True)
            uploaded = colab_files.upload()
            if uploaded:
                for fname in uploaded:
                    print(f"✅  Uploaded: {fname}")
                new_opts = refresh_json_list()
                w_json.options = new_opts
                w_json.value = new_opts[0]
                load_deal2()
    w_upload_btn2.on_click(_colab_upload2)
else:
    w_upload_btn2 = widgets.FileUpload(
        accept='.json', multiple=False,
        description='📤  Upload JSON',
        layout=widgets.Layout(width='200px')
    )
    def _local_upload2(change):
        with upload_out2:
            clear_output(wait=True)
            for fname, file_info in w_upload_btn2.value.items():
                content = file_info['content']
                with open(fname, 'wb') as f:
                    f.write(content)
                print(f"✅  Uploaded: {fname}")
            new_opts = refresh_json_list()
            w_json.options = new_opts
            w_json.value = new_opts[0]
            load_deal2()
    w_upload_btn2.observe(_local_upload2, names='value')

display(widgets.VBox([
    widgets.HTML('<b style="color:#58a6ff">▸ Source: lng_feasibility_extended.ipynb → Section 7 export</b>'),
    widgets.HBox([w_json, w_load_btn2]),
    widgets.HTML('<b style="color:#8b949e;font-size:11px">▸ Or upload a JSON file from your computer</b>'),
    widgets.HBox([w_upload_btn2]),
    upload_out2,
    load_out2,
    widgets.HTML('<b style="color:#8b949e;font-size:11px">▸ Manual overrides</b>'),
    w_net_mmbtu2, w_buy_px2, w_sell_px2,
    w_cpty2, w_buy_cpty2, w_total_voyage, w_regas2,
    w_bog_rate2, w_voyage_days2,
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))
if all_jsons[0] != 'No JSON found':
    load_deal2()


---
## 📋 Phase 2 — Bill of Lading & Title Transfer
*Enter BL data from the loading port. This triggers the first inventory journal entry.*

In [ ]:
# ── CELL 3: Bill of Lading ────────────────────────────────────────────────────
w_bl_date        = widgets.DatePicker(value=date.today() + timedelta(days=18),
    description='BL Date:', style=style, layout=layout)
w_bl_number      = widgets.Text(value='RLNG-2026-0419-PP',
    description='BL Number:', style=style, layout=layout)
w_loaded_vol_m3  = widgets.FloatText(value=137842,
    description='Loaded Volume (m³):', style=style, layout=layout)
w_density_bl     = widgets.FloatText(value=0.4601,
    description='LNG Density (t/m³):', style=style, layout=layout)
w_cv_bl          = widgets.FloatText(value=21.52,
    description='Calorific Value (MMBtu/t):', style=style, layout=layout)
w_delivery_terms = widgets.Dropdown(
    options=['FOB (title at load port)', 'DES (title at discharge port)'],
    value='FOB (title at load port)',
    description='Delivery Terms:', style=style, layout=layout
)
w_vessel_name    = widgets.Text(value='Pacific Pearl',
    description='Vessel Name:', style=style, layout=layout)

bl_out = widgets.Output()

def run_bl(_=None):
    JOURNAL.clear()
    vol_m3    = w_loaded_vol_m3.value
    dens      = w_density_bl.value
    cv        = w_cv_bl.value
    buy_px    = w_buy_px2.value
    weight    = vol_m3 * dens
    gross_mmbtu = weight * cv
    ap_amount = gross_mmbtu * buy_px
    is_fob    = 'FOB' in w_delivery_terms.value

    with bl_out:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
        fig.patch.set_facecolor(C['bg'])

        # Volume summary chart
        ax1.set_facecolor(C['card'])
        cats_ = ['Vessel\nCapacity (m³)', 'Loaded\nVolume (m³)',
                 'Gross Energy\n(MMBtu÷1000)', 'BL AP\n($M)']
        vals_ = [w_net_mmbtu2.value / (dens * cv) * w_density_bl.value,
                 vol_m3, gross_mmbtu/1000, ap_amount/1e6]
        ax1.bar(cats_, vals_, color=[C['muted'], C['blue'],
                                      C['gain'], C['warn']],
                alpha=0.85, edgecolor='#21262d', linewidth=0.5)
        for i, v in enumerate(vals_):
            ax1.text(i, v + max(vals_)*0.02, f'{v:,.1f}',
                     ha='center', fontsize=9, fontweight='bold',
                     color=C['text'])
        ax1.set_title(f'BL Summary — {w_bl_number.value}', fontsize=10)
        ax1.grid(True, axis='y', alpha=0.3)

        # Journal entry visual
        ax2.set_facecolor(C['card'])
        ax2.set_xticks([]); ax2.set_yticks([])
        for sp in ax2.spines.values(): sp.set_edgecolor(C['border'])

        if is_fob:
            title_note = "Title transfers to PacificLNG at loading port"
            je_entries = [
                ('DR', 'LNG Inventory — Loaded (1300)',
                 ap_amount, 'Asset created on title transfer'),
                ('CR', f'Accounts Payable — {w_buy_cpty2.value} (2100)',
                 ap_amount, f'AP to {w_buy_cpty2.value} due {(w_bl_date.value + timedelta(days=5)).strftime("%d %b") if w_bl_date.value else "T+5"}'),
            ]
        else:
            title_note = "Title transfers at DISCHARGE — no inventory entry yet"
            je_entries = [
                ('MEMO', 'No GL entry at loading for DES', 0,
                 'Seller retains inventory until discharge'),
            ]

        ax2.text(0.5, 0.92, title_note, transform=ax2.transAxes,
                 ha='center', fontsize=8.5, color=C['warn'], style='italic')

        for i, (dr_cr, acct, amt, narr) in enumerate(je_entries):
            ypos = 0.72 - i * 0.22
            col  = C['dr'] if dr_cr == 'DR' else                    C['cr'] if dr_cr == 'CR' else C['muted']
            ax2.text(0.02, ypos, f'{dr_cr}:  {acct}',
                     transform=ax2.transAxes, fontsize=8.5, color=col)
            if amt > 0:
                ax2.text(0.72, ypos, f'${amt:,.0f}',
                         transform=ax2.transAxes, fontsize=8.5,
                         color=col, fontweight='bold')
            ax2.text(0.02, ypos - 0.06, f'     {narr}',
                     transform=ax2.transAxes, fontsize=7.5,
                     color=C['muted'], style='italic')

        ax2.set_title('Accounting Journal — BL Event', fontsize=10)

        fig.suptitle(
            f'Bill of Lading — {w_vessel_name.value} — {w_bl_date.value}',
            fontsize=11, fontweight='bold', color=C['text']
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Post to journal
        if is_fob:
            add_journal('Loading/BL', 'LNG Inventory — Loaded (1300)',
                        'DR', ap_amount, '1300',
                        f'Cargo loaded {w_vessel_name.value} BL#{w_bl_number.value}')
            add_journal('Loading/BL', f'AP — {w_buy_cpty2.value} (2100)',
                        'CR', ap_amount, '2100',
                        f'AP to {w_buy_cpty2.value} due T+5 from BL')
            # Reclassify to in-transit
            add_journal('Loading/BL', 'LNG Inventory — In-Transit (1310)',
                        'DR', ap_amount, '1310', 'Vessel departed — reclassify')
            add_journal('Loading/BL', 'LNG Inventory — Loaded (1300)',
                        'CR', ap_amount, '1300', 'Reclassify loaded → in-transit')

        print(f"  BL Number:       {w_bl_number.value}")
        print(f"  BL Date:         {w_bl_date.value}")
        print(f"  Vessel:          {w_vessel_name.value}")
        print(f"  Loaded (m³):     {vol_m3:,.0f}")
        print(f"  Weight (t):      {weight:,.0f}")
        print(f"  Gross Energy:    {gross_mmbtu:,.0f} MMBtu")
        print(f"  Buy Price:       ${buy_px:.4f}/MMBtu")
        print(f"  AP Created:      ${ap_amount:,.0f}")
        print(f"  Title Transfer:  {w_delivery_terms.value}")
        print(f"  Journal entries posted: {len(JOURNAL)}")

w_bl_btn = widgets.Button(
    description='▶  Process BL & Post Inventory',
    button_style='warning',
    layout=widgets.Layout(width='260px', height='38px')
)
w_bl_btn.on_click(run_bl)

display(widgets.VBox([
    widgets.HTML('<b style="color:#d29922;font-size:13px">▸ Bill of Lading Data</b>'),
    w_bl_date, w_bl_number, w_vessel_name,
    w_loaded_vol_m3, w_density_bl, w_cv_bl,
    w_delivery_terms,
    w_bl_btn, bl_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## 🚢 Phase 3 — In-Transit Lifecycle & Accruals
*Daily BOG accrual, freight accrual by days-sailed, cross-month detection.*

In [ ]:
# ── CELL 4: In-transit lifecycle ──────────────────────────────────────────────
w_load_date  = widgets.DatePicker(value=date.today() + timedelta(days=18),
    description='Load Date (BL):', style=style, layout=layout)
w_est_disch  = widgets.DatePicker(value=date.today() + timedelta(days=38),
    description='Est. Discharge Date:', style=style, layout=layout)
w_period_end = widgets.DatePicker(value=date.today() + timedelta(days=30),
    description='Period End Date:', style=style, layout=layout)
w_freight_total = widgets.FloatText(value=3476258,
    description='Total Freight ($):', style=style, layout=layout)
w_inv_value  = widgets.FloatText(value=18124347,
    description='Inventory Cost (AP amount $):', style=style, layout=layout)

transit_out = widgets.Output()

def run_transit(_=None):
    load_dt    = w_load_date.value
    disch_dt   = w_est_disch.value
    period_dt  = w_period_end.value
    frt_total  = w_freight_total.value
    inv_val    = w_inv_value.value
    bog_rate   = w_bog_rate2.value / 100
    voyage_d   = (disch_dt - load_dt).days if (disch_dt and load_dt) else w_voyage_days2.value

    cross_month = (load_dt and disch_dt and
                   load_dt.month != disch_dt.month)
    accrual_needed = (period_dt and load_dt and disch_dt and
                      load_dt <= period_dt < disch_dt)

    days_sailed_at_period = max(0, (period_dt - load_dt).days) if (period_dt and load_dt) else 0
    days_remaining        = max(0, voyage_d - days_sailed_at_period)

    # BOG accrual to period end
    bog_pct_sailed = bog_rate * days_sailed_at_period
    bog_value      = inv_val * bog_pct_sailed

    # Freight accrual (proportion of days sailed)
    frt_accrual = frt_total * (days_sailed_at_period / voyage_d) if voyage_d > 0 else 0

    with transit_out:
        clear_output(wait=True)
        fig = plt.figure(figsize=(15, 9))
        fig.patch.set_facecolor(C['bg'])
        gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.32)

        # Chart 1: Voyage timeline
        ax1 = fig.add_subplot(gs[0, :])
        ax1.set_facecolor(C['card'])
        ax1.set_xlim(-0.5, voyage_d + 2)
        ax1.set_ylim(0, 2.5)

        # Timeline bar
        ax1.barh(1.2, voyage_d, color=C['blue'], alpha=0.3, height=0.6, left=0)
        ax1.barh(1.2, days_sailed_at_period, color=C['blue'],
                 alpha=0.8, height=0.6, left=0)
        if days_remaining > 0:
            ax1.barh(1.2, days_remaining, color=C['muted'],
                     alpha=0.35, height=0.6,
                     left=days_sailed_at_period)

        # Events
        events_tl = [
            (0, '📦\nLoaded', C['gain']),
            (days_sailed_at_period, '📅\nPeriod\nEnd', C['warn']),
            (voyage_d, '⚓\nDischarged', C['teal']),
        ]
        for x_pos, lbl, col in events_tl:
            ax1.axvline(x_pos, color=col, lw=1.5, ls='--', alpha=0.8)
            ax1.text(x_pos, 2.1, lbl, ha='center', fontsize=7.5,
                     color=col, va='bottom')

        ax1.set_xlabel('Days since Loading', fontsize=9)
        ax1.set_yticks([])
        cross_str = '⚠️ CROSS-MONTH CARGO' if cross_month else '✅ Same-month cargo'
        cross_col = C['warn'] if cross_month else C['gain']
        ax1.set_title(
            f'Voyage Timeline — {voyage_d} days | {cross_str}',
            fontsize=10, color=cross_col
        )
        ax1.text(days_sailed_at_period/2, 1.2,
                 f'{days_sailed_at_period}d sailed', ha='center',
                 va='center', fontsize=8, color=C['bg'], fontweight='bold')

        # Chart 2: BOG accumulation
        ax2 = fig.add_subplot(gs[1, 0])
        ax2.set_facecolor(C['card'])
        days_range = np.arange(0, voyage_d + 1)
        bog_vals   = [inv_val * bog_rate * d for d in days_range]
        ax2.plot(days_range, [v/1000 for v in bog_vals],
                 color=C['loss'], lw=2, label='BOG Loss ($000s)')
        ax2.axvline(days_sailed_at_period, color=C['warn'],
                    lw=1.5, ls='--',
                    label=f'Period end: ${bog_value/1000:.1f}k')
        ax2.fill_between(days_range[:days_sailed_at_period+1],
                         [v/1000 for v in bog_vals[:days_sailed_at_period+1]],
                         alpha=0.3, color=C['loss'])
        ax2.set_xlabel('Voyage Day', fontsize=8)
        ax2.set_ylabel('BOG Loss ($000s)', fontsize=8)
        ax2.set_title('Cumulative BOG Accrual', fontsize=10)
        ax2.legend(fontsize=7.5)
        ax2.grid(True, alpha=0.3)

        # Chart 3: Freight accrual
        ax3 = fig.add_subplot(gs[1, 1])
        ax3.set_facecolor(C['card'])
        frt_daily = frt_total / voyage_d if voyage_d > 0 else 0
        frt_cum   = [frt_daily * d for d in days_range]
        ax3.plot(days_range, [v/1e6 for v in frt_cum],
                 color=C['warn'], lw=2)
        ax3.axvline(days_sailed_at_period, color=C['warn'],
                    lw=1.5, ls='--',
                    label=f'Accrued: ${frt_accrual/1e6:.2f}M')
        ax3.fill_between(days_range[:days_sailed_at_period+1],
                         [v/1e6 for v in frt_cum[:days_sailed_at_period+1]],
                         alpha=0.3, color=C['warn'])
        ax3.set_xlabel('Voyage Day', fontsize=8)
        ax3.set_ylabel('Freight Accrual ($M)', fontsize=8)
        ax3.set_title('Freight Accrual by Day', fontsize=10)
        ax3.legend(fontsize=7.5)
        ax3.grid(True, alpha=0.3)

        # Period-end journal entries
        ax4 = fig.add_subplot(gs[1, 2])
        ax4.set_facecolor(C['card'])
        ax4.set_xticks([]); ax4.set_yticks([])
        for sp in ax4.spines.values(): sp.set_edgecolor(C['border'])

        ax4.text(0.5, 0.96, 'PERIOD-END ACCRUALS',
                 transform=ax4.transAxes, ha='center',
                 fontsize=9, fontweight='bold', color=C['text'])

        entries_4 = []
        if accrual_needed:
            entries_4 = [
                ('DR', 'Freight Expense (5200)',     frt_accrual,  '5200'),
                ('CR', 'Accrued Freight Pay (2200)', frt_accrual,  '2200'),
                ('',   '',                            0,             ''),
                ('DR', 'BOG Loss Expense (5300)',    bog_value,    '5300'),
                ('CR', 'LNG Inv — In-Transit (1310)',bog_value,    '1310'),
            ]
        else:
            entries_4 = [('INFO', 'No cross-month accrual required', 0, '')]

        for i, (dc, acct, amt, gl) in enumerate(entries_4):
            ypos = 0.82 - i * 0.14
            col  = C['dr'] if dc == 'DR' else                    C['cr'] if dc == 'CR' else C['muted']
            ax4.text(0.02, ypos, f'{dc}  {acct}',
                     transform=ax4.transAxes, fontsize=7.5, color=col)
            if amt > 0:
                ax4.text(0.85, ypos, f'${amt:,.0f}',
                         transform=ax4.transAxes,
                         fontsize=7.5, color=col,
                         fontweight='bold', ha='right')

        ax4.set_title('Period-End Journals', fontsize=10)

        fig.suptitle(
            f'In-Transit Lifecycle | Sailed: {days_sailed_at_period} of {voyage_d} days'
            + (' | ⚠️ CROSS-MONTH' if cross_month else ''),
            fontsize=11, fontweight='bold', color=C['text']
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Post accruals to journal
        if accrual_needed:
            add_journal('Period-End Accrual',
                        'Freight Expense (5200)', 'DR', frt_accrual,
                        '5200', f'{days_sailed_at_period}/{voyage_d} days sailed')
            add_journal('Period-End Accrual',
                        'Accrued Freight Payable (2200)', 'CR', frt_accrual,
                        '2200', 'Accrued freight proportion')
            add_journal('Period-End Accrual',
                        'BOG Loss Expense (5300)', 'DR', bog_value,
                        '5300', f'{days_sailed_at_period}d × {w_bog_rate2.value:.2f}%/d')
            add_journal('Period-End Accrual',
                        'LNG Inventory — In-Transit (1310)', 'CR', bog_value,
                        '1310', 'BOG reduces inventory value')

        print(f"  Voyage Duration:       {voyage_d} days")
        print(f"  Days Sailed at Period: {days_sailed_at_period} days")
        print(f"  Cross-Month:           {'YES ⚠️' if cross_month else 'NO'}")
        print(f"  Accrual Required:      {'YES' if accrual_needed else 'NO'}")
        print(f"  Freight Accrual:       ${frt_accrual:,.0f}")
        print(f"  BOG Accrual:           ${bog_value:,.0f}")
        print(f"  Total Period-End Accrual: ${frt_accrual + bog_value:,.0f}")

w_transit_btn = widgets.Button(
    description='▶  Calculate Accruals',
    button_style='primary',
    layout=widgets.Layout(width='220px', height='38px')
)
w_transit_btn.on_click(run_transit)
for w in [w_load_date, w_est_disch, w_period_end, w_bog_rate2]:
    w.observe(run_transit, names='value')

display(widgets.VBox([
    widgets.HTML('<b style="color:#bc8cff;font-size:13px">▸ Transit Configuration</b>'),
    w_load_date, w_est_disch, w_period_end,
    w_freight_total, w_inv_value, w_bog_rate2, w_voyage_days2,
    w_transit_btn, transit_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## ⚓ Phases 4 & 5 — Discharge, Outturn & Demurrage

In [ ]:
# ── CELL 5: Discharge, outturn & demurrage ────────────────────────────────────
w_nor_date   = widgets.DatePicker(value=date.today() + timedelta(days=38),
    description='NOR Tendered Date:', style=style, layout=layout)
w_nor_time   = widgets.FloatText(value=6.25,
    description='NOR Time (decimal hr, e.g. 6.25=06:15):', style=style, layout=layout)
w_pump_done  = widgets.FloatText(value=40.5,
    description='Hours from NOR to Pump Done:', style=style, layout=layout)
w_cosp_add   = widgets.FloatText(value=1.5,
    description='Hours: Pump Done to COSP:', style=style, layout=layout)
w_allowed_lt = widgets.FloatText(value=36.0,
    description='Allowed Laytime (hours):', style=style, layout=layout)
w_dem_rate   = widgets.FloatText(value=78000,
    description='Demurrage Rate ($/day):', style=style, layout=layout)
w_disch_vol  = widgets.FloatText(value=136432,
    description='Discharged Volume (m³):', style=style, layout=layout)
w_density_d  = widgets.FloatText(value=0.4601,
    description='Density at Discharge (t/m³):', style=style, layout=layout)
w_cv_d       = widgets.FloatText(value=21.48,
    description='CV at Discharge (MMBtu/t):', style=style, layout=layout)
w_heel_d     = widgets.FloatText(value=1100,
    description='Heel Remaining in Ship (m³):', style=style, layout=layout)
w_final_index = widgets.FloatText(value=16.45,
    description='Final Index Price ($/MMBtu):', style=style, layout=layout)
w_sell_discount = widgets.FloatText(value=-0.30,
    description='Sell Discount vs Index ($/MMBtu):', style=style, layout=layout)

discharge_out = widgets.Output()

def run_discharge(_=None):
    disch_m3    = w_disch_vol.value
    dens_d      = w_density_d.value
    cv_d        = w_cv_d.value
    heel_m3     = w_heel_d.value
    heel_mmbtu  = heel_m3 * dens_d * cv_d
    weight_d    = disch_m3 * dens_d
    outturn     = weight_d * cv_d
    buy_px      = w_buy_px2.value
    inv_val     = w_inv_value.value

    sell_px_final = w_final_index.value + w_sell_discount.value
    revenue_final = outturn * sell_px_final

    # BOG reconciliation
    gross_loaded  = w_net_mmbtu2.value   # approximation
    total_bog     = gross_loaded - outturn
    bog_pct       = total_bog / gross_loaded * 100 if gross_loaded > 0 else 0
    bog_val_final = total_bog * buy_px

    # Laytime & demurrage
    actual_lt     = w_pump_done.value + w_cosp_add.value
    excess        = max(0, actual_lt - w_allowed_lt.value)
    demurrage_amt = excess / 24 * w_dem_rate.value

    cogs          = inv_val - bog_val_final
    gross_profit  = revenue_final - cogs
    voyage_costs  = w_total_voyage.value
    regas_costs   = w_regas2.value
    net_profit    = gross_profit - voyage_costs - regas_costs + demurrage_amt

    with discharge_out:
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 2, figsize=(15, 9))
        fig.patch.set_facecolor(C['bg'])
        axes = axes.flatten()

        # 1: Volume reconciliation
        ax = axes[0]; ax.set_facecolor(C['card'])
        cats_  = ['Loaded\n(Gross)', 'BOG &\nHeel Lost',
                  'Outturn\n(Delivered)']
        vals_v = [gross_loaded, -(total_bog), outturn]
        cols_v = [C['blue'], C['loss'], C['gain']]
        ax.bar(cats_, [abs(v) for v in vals_v], color=cols_v,
               alpha=0.85, edgecolor='#21262d')
        for i, v in enumerate(vals_v):
            ax.text(i, abs(v) + gross_loaded*0.01,
                    f'{abs(v)/1e6:.3f}M\nMMBtu',
                    ha='center', fontsize=8.5,
                    fontweight='bold', color=C['text'])
        ax.set_title(f'Volume Reconciliation (BOG: {bog_pct:.2f}%)', fontsize=10)
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'{v/1e6:.2f}M'))
        ax.grid(True, axis='y', alpha=0.3)

        # 2: Laytime gauge
        ax2 = axes[1]; ax2.set_facecolor(C['card'])
        color_ = C['gain'] if excess == 0 else C['loss']
        ax2.barh([''], [w_allowed_lt.value], color=C['muted'],
                 alpha=0.5, height=0.4)
        ax2.barh([''], [min(actual_lt, w_allowed_lt.value)],
                 color=C['gain'], alpha=0.85, height=0.4)
        if excess > 0:
            ax2.barh([''], [excess], left=w_allowed_lt.value,
                     color=C['loss'], alpha=0.85, height=0.4)
        ax2.axvline(w_allowed_lt.value, color=C['warn'],
                    lw=2, ls='--',
                    label=f'Allowed: {w_allowed_lt.value:.0f}h')
        ax2.set_xlabel('Hours', fontsize=8)
        ax2.set_xlim(0, max(actual_lt, w_allowed_lt.value) * 1.15)
        ax2.set_yticks([])
        ax2.legend(fontsize=8)
        ax2.set_title(
            f'Laytime — Actual: {actual_lt:.1f}h | '
            f'{"Demurrage: ${:,.0f}".format(demurrage_amt) if excess > 0 else "On Laytime ✅"}',
            fontsize=10,
            color=C['loss'] if excess > 0 else C['gain']
        )
        ax2.grid(True, axis='x', alpha=0.3)

        # 3: Final P&L waterfall
        ax3 = axes[2]; ax3.set_facecolor(C['card'])
        wf_l = ['Revenue', 'COGS', 'Gross\nProfit',
                'Voyage\nCosts', 'Regas', 'Demurrage', 'NET P&L']
        wf_v = [revenue_final, -cogs, gross_profit,
                -voyage_costs, -regas_costs,
                demurrage_amt, net_profit]
        wf_c = [C['blue'], C['loss'], C['blue'],
                C['loss'], C['loss'],
                C['gain'] if demurrage_amt >= 0 else C['loss'],
                C['gain'] if net_profit >= 0 else C['loss']]
        ax3.bar(range(len(wf_l)),
                [v/1e6 for v in wf_v], color=wf_c,
                alpha=0.85, edgecolor='#21262d', linewidth=0.5)
        for i, v in enumerate(wf_v):
            ax3.text(i, v/1e6 + max(abs(x) for x in wf_v)/1e6*0.04,
                     f'${v/1e6:+.2f}M', ha='center', fontsize=7.5,
                     fontweight='bold', color=C['text'])
        ax3.set_xticks(range(len(wf_l)))
        ax3.set_xticklabels(wf_l, fontsize=8)
        ax3.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax3.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.1f}M'))
        ax3.set_title('Final Cargo P&L', fontsize=10)
        ax3.grid(True, axis='y', alpha=0.3)

        # 4: Journal entries
        ax4 = axes[3]; ax4.set_facecolor(C['card'])
        ax4.set_xticks([]); ax4.set_yticks([])
        for sp in ax4.spines.values(): sp.set_edgecolor(C['border'])
        je_ = [
            ('DR', f'AR — {w_cpty2.value} (1200)',
             revenue_final, 'Revenue on discharge'),
            ('CR', 'Revenue — LNG DES (4100)',
             revenue_final, 'Revenue recognised'),
            ('DR', 'COGS — LNG (5100)',   cogs,   'COGS recognition'),
            ('CR', 'LNG Inv — In-Transit (1310)', cogs, 'Clear inventory'),
            ('DR', 'BOG Loss (5300)',     bog_val_final, 'Actual BOG'),
            ('CR', 'LNG Inv — In-Transit (1310)', bog_val_final, 'BOG adj'),
        ]
        if demurrage_amt > 0:
            je_ += [
                ('DR', 'Demurrage Receivable (1250)', demurrage_amt, 'Demurrage claim'),
                ('CR', 'Demurrage Income (4150)',      demurrage_amt, 'Demurrage income'),
            ]
        ax4.text(0.5, 0.97, 'DISCHARGE JOURNAL ENTRIES', ha='center',
                 transform=ax4.transAxes, fontsize=8.5,
                 fontweight='bold', color=C['text'])
        for i, (dc, acct, amt, narr) in enumerate(je_):
            ypos = 0.88 - i * 0.10
            col  = C['dr'] if dc == 'DR' else C['cr']
            ax4.text(0.01, ypos, f'{dc}  {acct}',
                     transform=ax4.transAxes, fontsize=7.5, color=col)
            ax4.text(0.82, ypos, f'${amt:,.0f}',
                     transform=ax4.transAxes, fontsize=7.5,
                     color=col, fontweight='bold', ha='right')
        ax4.set_title('Journal Entries — Discharge', fontsize=10)

        verdict = '✅ PROFIT' if net_profit > 0 else '❌ LOSS'
        fig.suptitle(
            f'Discharge — {w_nor_date.value} | Outturn: {outturn:,.0f} MMBtu | '
            f'Net P&L: ${net_profit/1e6:+.2f}M | {verdict}',
            fontsize=11, fontweight='bold',
            color=C['gain'] if net_profit > 0 else C['loss']
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        # Post to journal
        add_journal('Discharge', f'AR — {w_cpty2.value} (1200)',
                    'DR', revenue_final, '1200', 'Revenue on DES delivery')
        add_journal('Discharge', 'Revenue — LNG DES (4100)',
                    'CR', revenue_final, '4100', 'Revenue recognised')
        add_journal('Discharge', 'COGS — LNG (5100)',
                    'DR', cogs, '5100', 'COGS on outturn')
        add_journal('Discharge', 'LNG Inventory — In-Transit (1310)',
                    'CR', cogs, '1310', 'Clear inventory on delivery')
        add_journal('Discharge', 'BOG Loss Expense (5300)',
                    'DR', bog_val_final, '5300', 'Total voyage BOG')
        add_journal('Discharge', 'LNG Inventory — In-Transit (1310)',
                    'CR', bog_val_final, '1310', 'BOG adjustment')
        if demurrage_amt > 0:
            add_journal('Demurrage', 'Demurrage Receivable (1250)',
                        'DR', demurrage_amt, '1250', f'{excess:.2f}h excess laytime')
            add_journal('Demurrage', 'Demurrage Income (4150)',
                        'CR', demurrage_amt, '4150', 'Demurrage claim on counterparty')

        print(f"  Outturn Volume:     {outturn:,.0f} MMBtu")
        print(f"  Final Sell Price:   ${sell_px_final:.4f}/MMBtu")
        print(f"  Revenue:            ${revenue_final:,.0f}")
        print(f"  COGS:               ${cogs:,.0f}")
        print(f"  Gross Profit:       ${gross_profit:+,.0f}")
        print(f"  Total BOG:          {total_bog:,.0f} MMBtu ({bog_pct:.2f}%)")
        print(f"  Actual Laytime:     {actual_lt:.1f}h vs {w_allowed_lt.value:.0f}h allowed")
        print(f"  Demurrage:          ${demurrage_amt:,.0f}")
        print(f"  NET P&L:            ${net_profit:+,.0f}")

w_disch_btn = widgets.Button(
    description='▶  Process Discharge & Outturn',
    button_style='danger',
    layout=widgets.Layout(width='260px', height='38px')
)
w_disch_btn.on_click(run_discharge)

display(widgets.VBox([
    widgets.HTML('<b style="color:#f85149;font-size:13px">▸ Discharge Data</b>'),
    w_nor_date, w_nor_time, w_pump_done, w_cosp_add,
    w_allowed_lt, w_dem_rate,
    widgets.HTML('<b style="color:#f85149;font-size:13px">▸ Outturn Measurements</b>'),
    w_disch_vol, w_density_d, w_cv_d, w_heel_d,
    widgets.HTML('<b style="color:#f85149;font-size:13px">▸ Final Pricing</b>'),
    w_final_index, w_sell_discount,
    w_disch_btn, discharge_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## 🧾 Phase 6 — Invoice Generation
*Builds the formal invoice from outturn data and posts AR.*

In [ ]:
# ── CELL 6: Invoice generation ───────────────────────────────────────────────
w_inv_num     = widgets.Text(value='PLNG-INV-2026-00489',
    description='Invoice Number:', style=style, layout=layout)
w_inv_date    = widgets.DatePicker(value=date.today() + timedelta(days=40),
    description='Invoice Date:', style=style, layout=layout)
w_payment_days_inv = widgets.IntSlider(value=5, min=0, max=30,
    description='Payment Terms (days):', style=style, layout=layout)
w_invoice_vol = widgets.FloatText(value=1348847,
    description='Invoice Volume (MMBtu):', style=style, layout=layout)
w_invoice_px  = widgets.FloatText(value=16.15,
    description='Invoice Price ($/MMBtu):', style=style, layout=layout)
w_invoice_dem = widgets.FloatText(value=13813,
    description='Demurrage on Invoice ($):', style=style, layout=layout)
w_vat_rate    = widgets.FloatSlider(value=0.0, min=0.0, max=25.0, step=0.5,
    description='VAT/Tax Rate (%):', style=style, layout=layout,
    readout_format='.1f')

invoice_out = widgets.Output()

def run_invoice(_=None):
    vol       = w_invoice_vol.value
    px        = w_invoice_px.value
    dem       = w_invoice_dem.value
    vat_rate  = w_vat_rate.value / 100
    commodity = vol * px
    vat_amt   = (commodity + dem) * vat_rate
    total_inv = commodity + dem + vat_amt
    inv_date  = w_inv_date.value
    due_date  = inv_date + timedelta(days=w_payment_days_inv.value) if inv_date else None

    # Payable to buy-side (AP to QE)
    buy_vol   = w_net_mmbtu2.value
    buy_amt   = buy_vol * w_buy_px2.value

    with invoice_out:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        fig.patch.set_facecolor(C['bg'])

        # Invoice layout
        ax1.set_facecolor('#0d1520')
        ax1.set_xlim(0, 10); ax1.set_ylim(0, 14)
        ax1.set_xticks([]); ax1.set_yticks([])
        for sp in ax1.spines.values(): sp.set_edgecolor(C['blue'])
        ax1.spines['top'].set_linewidth(3)
        ax1.spines['bottom'].set_linewidth(3)

        ax1.text(5, 13.2, 'TAX INVOICE', ha='center', fontsize=16,
                 fontweight='bold', color=C['blue'])
        ax1.text(5, 12.6, 'PacificLNG Trading Co. (Singapore)',
                 ha='center', fontsize=9, color=C['muted'])
        ax1.axhline(12.2, color=C['blue'], lw=1.5, alpha=0.5)

        inv_fields = [
            ('Invoice Number:', w_inv_num.value),
            ('Invoice Date:',   str(inv_date) if inv_date else ''),
            ('Due Date:',       str(due_date) if due_date else f'Net {w_payment_days_inv.value} days'),
            ('Bill To:',        w_cpty2.value),
            ('Vessel:',         w_vessel_name.value if 'w_vessel_name' in dir() else 'Pacific Pearl'),
        ]
        for i, (lbl, val) in enumerate(inv_fields):
            y = 11.8 - i * 0.55
            ax1.text(0.3, y, lbl, fontsize=8.5, color=C['muted'])
            ax1.text(4.0, y, val, fontsize=8.5, color=C['text'], fontweight='bold')

        ax1.axhline(8.8, color=C['border'], lw=0.8)
        line_items = [
            ('LNG Cargo — DES Delivery',
             f'{vol:,.0f} MMBtu', f'${px:.4f}',  commodity),
            ('Demurrage Charge', '—', '—', dem),
        ]
        if vat_amt > 0:
            line_items.append(('VAT / Tax', '—', f'{w_vat_rate.value:.1f}%', vat_amt))

        headers_ = ['Description', 'Volume', 'Unit Price', 'Amount']
        for j, h in enumerate(headers_):
            xp = [0.3, 4.2, 6.5, 8.0][j]
            ax1.text(xp, 8.5, h, fontsize=7.5,
                     color=C['muted'], fontweight='bold')

        for i, (desc, vol_, upx, amt) in enumerate(line_items):
            y = 7.9 - i * 0.65
            ax1.text(0.3,  y, desc, fontsize=8,   color=C['text'])
            ax1.text(4.2,  y, vol_, fontsize=8,   color=C['text'])
            ax1.text(6.5,  y, upx,  fontsize=8,   color=C['text'])
            ax1.text(9.7,  y, f'${amt:,.0f}',
                     fontsize=8, color=C['text'], ha='right')

        ax1.axhline(5.8, color=C['border'], lw=0.8)
        ax1.text(7.0, 5.4, 'TOTAL DUE:', fontsize=10,
                 color=C['text'], fontweight='bold')
        ax1.text(9.7, 5.4, f'${total_inv:,.0f}',
                 fontsize=10, color=C['gain'],
                 fontweight='bold', ha='right')
        ax1.text(5, 4.4, f'Payment due: {due_date}  |  Wire transfer USD',
                 ha='center', fontsize=8, color=C['muted'])
        ax1.set_title(f'Invoice #{w_inv_num.value}', fontsize=10)

        # AR/AP balance chart
        ax2.set_facecolor(C['card'])
        items  = [f'AR — {w_cpty2.value[:12]}', f'AP — {w_buy_cpty2.value[:12]}', 'Net Position']
        values = [total_inv, -buy_amt, total_inv - buy_amt]
        colors = [C['gain'], C['loss'],
                  C['gain'] if (total_inv - buy_amt) >= 0 else C['loss']]
        bars__ = ax2.bar(items, [v/1e6 for v in values],
                         color=colors, alpha=0.85,
                         edgecolor='#21262d', linewidth=0.5, width=0.5)
        for bar, val in zip(bars__, values):
            ax2.text(bar.get_x() + bar.get_width()/2,
                     bar.get_height()/1e6 +
                     abs(max(values, key=abs))/1e6 * 0.04,
                     f'${val/1e6:+.2f}M',
                     ha='center', fontsize=10,
                     fontweight='bold', color=C['text'])
        ax2.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.1f}M'))
        ax2.set_title('AR vs AP Position', fontsize=10)
        ax2.grid(True, axis='y', alpha=0.3)

        fig.suptitle(f'Invoice Generated — Total: ${total_inv:,.0f}  |  Due: {due_date}',
                     fontsize=11, fontweight='bold', color=C['gain'])
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(f"  Invoice:         {w_inv_num.value}")
        print(f"  Commodity:       ${commodity:,.0f}")
        print(f"  Demurrage:       ${dem:,.0f}")
        if vat_amt > 0:
            print(f"  VAT/Tax:         ${vat_amt:,.0f}")
        print(f"  TOTAL INVOICE:   ${total_inv:,.0f}")
        print(f"  Due Date:        {due_date}")
        print(f"  AP to {w_buy_cpty2.value}: ${buy_amt:,.0f}")
        print(f"  Net Cash Expected: ${total_inv - buy_amt:+,.0f}")

w_inv_btn = widgets.Button(
    description='▶  Generate Invoice',
    button_style='success',
    layout=widgets.Layout(width='200px', height='38px')
)
w_inv_btn.on_click(run_invoice)

display(widgets.VBox([
    widgets.HTML('<b style="color:#3fb950;font-size:13px">▸ Invoice Configuration</b>'),
    w_inv_num, w_inv_date, w_payment_days_inv,
    w_invoice_vol, w_invoice_px, w_invoice_dem, w_vat_rate,
    w_inv_btn, invoice_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## 💸 Phases 7 & 8 — GL Journal Ledger & Cash Settlement

In [ ]:
# ── CELL 7: Full GL ledger & settlement ──────────────────────────────────────
w_ar_received = widgets.FloatText(value=21797692,
    description='AR Cash Received ($):', style=style, layout=layout)
w_ap_paid     = widgets.FloatText(value=18124347,
    description='AP Cash Paid ($):', style=style, layout=layout)
w_freight_paid = widgets.FloatText(value=3476258,
    description='Freight AP Paid ($):', style=style, layout=layout)
w_netting_flag = widgets.Checkbox(value=False,
    description='Apply Netting (AR offset against AP)?',
    layout=widgets.Layout(width='400px'))

settlement_out = widgets.Output()

def run_settlement(_=None):
    ar_cash   = w_ar_received.value
    ap_cash   = w_ap_paid.value
    frt_cash  = w_freight_paid.value
    netting   = w_netting_flag.value
    net_cash  = ar_cash - ap_cash - frt_cash

    # Post settlement entries
    add_journal('AR Settlement', 'Cash / Bank (1100)',
                'DR', ar_cash, '1100', 'Cash received from buyer')
    add_journal('AR Settlement',
                f'AR — {w_cpty2.value} (1200)',
                'CR', ar_cash, '1200', 'AR cleared on payment receipt')
    add_journal('AP Settlement',
                f'AP — {w_buy_cpty2.value} (2100)',
                'DR', ap_cash, '2100', 'AP cleared — cargo payment')
    add_journal('AP Settlement', 'Cash / Bank (1100)',
                'CR', ap_cash, '1100', 'Cash paid to supplier')
    add_journal('Freight Settlement',
                'Accrued Freight Payable (2200)',
                'DR', frt_cash, '2200', 'Freight AP cleared')
    add_journal('Freight Settlement', 'Cash / Bank (1100)',
                'CR', frt_cash, '1100', 'Cash paid to ship owner')

    with settlement_out:
        clear_output(wait=True)
        fig = plt.figure(figsize=(16, 10))
        fig.patch.set_facecolor(C['bg'])
        gs = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3)

        # Cash flow summary
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.set_facecolor(C['card'])
        cf_items = ['AR Received\n(from buyer)',
                    'AP Paid\n(to supplier)',
                    'Freight Paid\n(to shipowner)',
                    'Net Cash\nPosition']
        cf_vals  = [ar_cash, -ap_cash, -frt_cash, net_cash]
        cf_cols  = [C['gain'], C['loss'], C['loss'],
                    C['gain'] if net_cash >= 0 else C['loss']]
        ax1.bar(range(4), [v/1e6 for v in cf_vals],
                color=cf_cols, alpha=0.85, edgecolor='#21262d')
        for i, v in enumerate(cf_vals):
            ax1.text(i, v/1e6 + abs(max(cf_vals,key=abs))/1e6*0.04,
                     f'${v/1e6:+.2f}M', ha='center',
                     fontsize=9, fontweight='bold', color=C['text'])
        ax1.set_xticks(range(4))
        ax1.set_xticklabels(cf_items, fontsize=8)
        ax1.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.1f}M'))
        ax1.set_title('Cash Settlement Summary', fontsize=10)
        ax1.grid(True, axis='y', alpha=0.3)

        # AR/AP lifecycle
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.set_facecolor(C['card'])
        stages = ['Created\n(BL/Discharge)', 'Sent\n(Invoice)', 'Matched', 'CLEARED\n(Paid)']
        for i, (stage, col) in enumerate(zip(stages,
            [C['muted'], C['warn'], C['blue'], C['gain']])):
            circ = plt.Circle((i, 0.5), 0.25, color=col,
                               alpha=0.85, zorder=3)
            ax2.add_patch(circ)
            ax2.text(i, 0.5, '✓', ha='center', va='center',
                     fontsize=12, fontweight='bold',
                     color=C['bg'], zorder=4)
            ax2.text(i, -0.05, stage, ha='center',
                     fontsize=7.5, color=C['text'])
            if i < 3:
                ax2.annotate('', xy=(i+0.75, 0.5),
                             xytext=(i+0.25, 0.5),
                             arrowprops=dict(
                                 arrowstyle='->', color=C['border'], lw=2)
                             )
        ax2.set_xlim(-0.5, 3.5); ax2.set_ylim(-0.3, 1.2)
        ax2.set_xticks([]); ax2.set_yticks([])
        ax2.set_title('AR Lifecycle — ALL STAGES COMPLETE ✅', fontsize=10)

        # Full GL ledger
        ax3 = fig.add_subplot(gs[1, :])
        ax3.set_facecolor(C['card'])
        ax3.set_xticks([]); ax3.set_yticks([])
        for sp in ax3.spines.values(): sp.set_edgecolor(C['border'])

        ax3.text(0.5, 0.97, 'COMPLETE GL JOURNAL — ALL PHASES',
                 transform=ax3.transAxes, ha='center',
                 fontsize=10, fontweight='bold', color=C['text'])

        df_j = pd.DataFrame(JOURNAL)
        events_unique = df_j['Event'].unique() if not df_j.empty else []
        col_map = {
            'Loading/BL':         C['blue'],
            'Period-End Accrual': C['warn'],
            'Discharge':          C['teal'],
            'Demurrage':          C['purple'],
            'AR Settlement':      C['gain'],
            'AP Settlement':      C['gain'],
            'Freight Settlement': C['gain'],
        }
        if not df_j.empty:
            y_start = 0.90
            dy = min(0.065, 0.90 / max(len(df_j), 1))
            for _, row in df_j.iterrows():
                col_ = col_map.get(row['Event'], C['muted'])
                dc_  = C['dr'] if row['DR/CR'] == 'DR' else C['cr']
                ax3.text(0.01, y_start,
                         f"[{row['Event'][:18]:<18}]",
                         transform=ax3.transAxes, fontsize=7,
                         color=col_, fontfamily='monospace')
                ax3.text(0.20, y_start,
                         f"{row['DR/CR']}  {row['Account'][:38]}",
                         transform=ax3.transAxes, fontsize=7,
                         color=dc_, fontfamily='monospace')
                ax3.text(0.80, y_start,
                         f"${row['Amount']:>14,.0f}",
                         transform=ax3.transAxes, fontsize=7,
                         color=dc_, ha='left', fontfamily='monospace')
                y_start -= dy
        else:
            ax3.text(0.5, 0.5, 'Run phases 2–7 to populate the journal.',
                     transform=ax3.transAxes, ha='center',
                     color=C['muted'], fontsize=10)
        ax3.set_title('', fontsize=10)

        fig.suptitle(
            f'Settlement Complete — Net Cash: ${net_cash/1e6:+.2f}M',
            fontsize=12, fontweight='bold',
            color=C['gain'] if net_cash >= 0 else C['loss']
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(f"  AR Received:   ${ar_cash:,.0f}")
        print(f"  AP Paid:       ${ap_cash:,.0f}")
        print(f"  Freight Paid:  ${frt_cash:,.0f}")
        print(f"  NET CASH:      ${net_cash:+,.0f}")
        print(f"  Total Journal Entries: {len(JOURNAL)}")
        print()
        show_journal()

w_sett_btn = widgets.Button(
    description='▶  Post Settlement & View Full GL',
    button_style='success',
    layout=widgets.Layout(width='280px', height='38px')
)
w_sett_btn.on_click(run_settlement)

display(widgets.VBox([
    widgets.HTML('<b style="color:#3fb950;font-size:13px">▸ Cash Settlement</b>'),
    w_ar_received, w_ap_paid, w_freight_paid, w_netting_flag,
    w_sett_btn, settlement_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## 📅 Phases 9 & 10 — Month-End Close & Final P&L Reconciliation

In [ ]:
# ── CELL 8: Month-end checklist & final P&L recon ────────────────────────────
monthend_out = widgets.Output()

w_me_ar_cleared  = widgets.Checkbox(value=False, description='AR fully cleared ($0 outstanding)')
w_me_ap_cleared  = widgets.Checkbox(value=False, description='AP fully cleared ($0 outstanding)')
w_me_frt_paid    = widgets.Checkbox(value=False, description='Freight invoice received & paid')
w_me_bog_done    = widgets.Checkbox(value=False, description='BOG reconciliation complete')
w_me_oci_zero    = widgets.Checkbox(value=False, description='OCI balance = $0 (reclassified on delivery)')
w_me_sap_done    = widgets.Checkbox(value=False, description='SAP interface: all journals posted')
w_me_emir_done   = widgets.Checkbox(value=False, description='EMIR/regulatory reports filed')
w_me_recon_done  = widgets.Checkbox(value=False, description='Endur/SAP reconciliation: PASS')
w_me_mtm_zero    = widgets.Checkbox(value=False, description='MTM balance = $0 (all positions settled)')
w_me_dem_settled = widgets.Checkbox(value=False, description='Demurrage claim settled/agreed')

all_checkboxes = [
    w_me_ar_cleared, w_me_ap_cleared, w_me_frt_paid,
    w_me_bog_done, w_me_oci_zero, w_me_sap_done,
    w_me_emir_done, w_me_recon_done, w_me_mtm_zero,
    w_me_dem_settled
]

me_out = widgets.Output()

def run_monthend(_=None):
    checks = [cb.value for cb in all_checkboxes]
    n_pass = sum(checks)
    n_total = len(checks)
    all_pass = all(checks)

    # Final P&L reconstruction
    rev   = w_invoice_vol.value * w_invoice_px.value
    cogs_ = w_ap_paid.value
    gp    = rev - cogs_
    ship_ = w_freight_paid.value
    regas_= w_regas2.value
    bog_e = (w_net_mmbtu2.value - w_invoice_vol.value) * w_buy_px2.value
    dem_i = w_invoice_dem.value
    net_p = gp - ship_ - regas_ - bog_e + dem_i

    with monthend_out:
        clear_output(wait=True)
        fig = plt.figure(figsize=(16, 9))
        fig.patch.set_facecolor(C['bg'])
        gs = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.32)

        # Checklist
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.set_facecolor(C['card'])
        ax1.set_xlim(0, 10); ax1.set_ylim(0, n_total + 1)
        ax1.set_xticks([]); ax1.set_yticks([])
        ax1.text(5, n_total + 0.5, 'MONTH-END CLOSE CHECKLIST',
                 ha='center', fontsize=9, fontweight='bold',
                 color=C['text'])
        for i, (cb, ok) in enumerate(zip(all_checkboxes, checks)):
            ypos = n_total - i - 0.3
            icon = '✅' if ok else '❌'
            col  = C['gain'] if ok else C['loss']
            ax1.text(0.5, ypos, icon, fontsize=10)
            ax1.text(1.5, ypos, cb.description,
                     fontsize=7.5, color=col, va='center')
        ax1.set_title(
            f'Checklist: {n_pass}/{n_total} Passed '
            f'{"✅ LOCK PERIOD" if all_pass else "⏳ PENDING"}',
            fontsize=10,
            color=C['gain'] if all_pass else C['warn']
        )

        # Progress donut
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.set_facecolor(C['card'])
        ax2.pie([n_pass, n_total - n_pass],
                colors=[C['gain'], C['border']],
                startangle=90,
                wedgeprops={'edgecolor': C['bg'], 'linewidth': 3},
                radius=1.0)
        ax2.pie([1], colors=[C['card']], radius=0.65)
        ax2.text(0, 0, f'{n_pass}/{n_total}',
                 ha='center', va='center',
                 fontsize=22, fontweight='bold',
                 color=C['gain'] if all_pass else C['warn'])
        ax2.set_title(
            '✅ ALL CLEAR — LOCK PERIOD' if all_pass else '⏳ Checklist Incomplete',
            fontsize=11, fontweight='bold',
            color=C['gain'] if all_pass else C['warn']
        )

        # Final P&L
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.set_facecolor(C['card'])
        pnl_items = ['Revenue', 'COGS', 'Freight', 'Regas',
                     'BOG Exp', 'Demurrage', 'NET P&L']
        pnl_vals  = [rev, -cogs_, -ship_, -regas_, -bog_e, dem_i, net_p]
        pnl_cols  = [C['blue'], C['loss'], C['loss'], C['loss'],
                     C['loss'], C['gain'], C['gain'] if net_p >= 0 else C['loss']]
        ax3.bar(range(len(pnl_items)),
                [v/1e6 for v in pnl_vals],
                color=pnl_cols, alpha=0.85,
                edgecolor='#21262d', linewidth=0.5)
        for i, v in enumerate(pnl_vals):
            ax3.text(i, v/1e6 + abs(max(pnl_vals,key=abs))/1e6*0.04,
                     f'${v/1e6:+.2f}M',
                     ha='center', fontsize=7.5,
                     fontweight='bold', color=C['text'])
        ax3.set_xticks(range(len(pnl_items)))
        ax3.set_xticklabels(pnl_items, fontsize=8)
        ax3.axhline(0, color=C['border'], lw=0.8, ls='--')
        ax3.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:.1f}M'))
        ax3.set_title('Final Realised P&L', fontsize=10)
        ax3.grid(True, axis='y', alpha=0.3)

        # Endur vs SAP reconciliation table
        ax4 = fig.add_subplot(gs[1, 1])
        ax4.set_facecolor(C['card'])
        ax4.set_xticks([]); ax4.set_yticks([])
        for sp in ax4.spines.values(): sp.set_edgecolor(C['border'])
        recon_items = [
            ('Revenue',     rev,    rev,    True),
            ('COGS',        cogs_,  cogs_,  True),
            ('AR Balance',  0,      0,      True),
            ('AP Balance',  0,      0,      True),
            ('MTM Balance', 0,      0,      True),
            ('OCI Balance', 0,      0,      True),
        ]
        ax4.text(0.5, 0.97, 'ENDUR vs SAP RECONCILIATION',
                 transform=ax4.transAxes, ha='center',
                 fontsize=8.5, fontweight='bold', color=C['text'])
        ax4.text(0.02, 0.89, 'Line Item', transform=ax4.transAxes,
                 fontsize=7.5, color=C['muted'], fontweight='bold')
        ax4.text(0.50, 0.89, 'Endur', transform=ax4.transAxes,
                 fontsize=7.5, color=C['muted'], fontweight='bold')
        ax4.text(0.72, 0.89, 'SAP', transform=ax4.transAxes,
                 fontsize=7.5, color=C['muted'], fontweight='bold')
        ax4.text(0.88, 0.89, 'Diff', transform=ax4.transAxes,
                 fontsize=7.5, color=C['muted'], fontweight='bold')
        for i, (lbl, endur_v, sap_v, match) in enumerate(recon_items):
            y = 0.79 - i * 0.11
            col = C['gain'] if match else C['loss']
            icon = '✅' if match else '❌'
            ax4.text(0.02, y, lbl,           transform=ax4.transAxes, fontsize=7.5, color=C['text'])
            ax4.text(0.45, y, f'${endur_v/1e6:.2f}M', transform=ax4.transAxes, fontsize=7.5, color=C['text'])
            ax4.text(0.67, y, f'${sap_v/1e6:.2f}M',   transform=ax4.transAxes, fontsize=7.5, color=C['text'])
            ax4.text(0.86, y, icon,           transform=ax4.transAxes, fontsize=8,   color=col)
        ax4.set_title('Recon — All Lines ✅' if all(r[3] for r in recon_items)
                      else 'Recon — BREAKS FOUND ❌', fontsize=10,
                      color=C['gain'])

        final_v = '✅ LOCK PERIOD' if all_pass else '⏳ COMPLETE CHECKLIST FIRST'
        fig.suptitle(
            f'Month-End Close — {final_v}  |  '
            f'Final Net P&L: ${net_p/1e6:+.2f}M',
            fontsize=12, fontweight='bold',
            color=C['gain'] if all_pass else C['warn']
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(f"  MONTH-END CLOSE SUMMARY")
        print(f"  {'═'*50}")
        print(f"  Checklist: {n_pass}/{n_total} passed")
        print(f"  FINAL CARGO P&L:")
        for lbl, val in zip(pnl_items, pnl_vals):
            print(f"    {lbl:<14} ${val:>+14,.0f}")
        print(f"  Period Status: {'✅ READY TO LOCK' if all_pass else '⚠️  DO NOT LOCK — checklist incomplete'}")

w_me_btn = widgets.Button(
    description='▶  Run Month-End Close',
    button_style='primary',
    layout=widgets.Layout(width='220px', height='38px')
)
w_me_btn.on_click(run_monthend)

display(widgets.VBox([
    widgets.HTML('<b style="color:#58a6ff;font-size:13px">▸ Month-End Checklist — tick each item when confirmed</b>'),
    widgets.HBox([
        widgets.VBox(all_checkboxes[:5]),
        widgets.VBox(all_checkboxes[5:])
    ], layout=widgets.Layout(gap='40px')),
    w_me_btn, monthend_out
], layout=widgets.Layout(padding='12px', border='1px solid #30363d', border_radius='8px')))


---
## 📌 Lifecycle Phases — Quick Reference

| Phase | Notebook Cell | Key Output | GL Accounts |
|---|---|---|---|
| BL & Title Transfer | Phase 2 | Inventory created | DR 1300 / CR 2100 |
| In-Transit Accruals | Phase 3 | Period-end BOG + freight | DR 5200,5300 / CR 2200,1310 |
| Discharge & Outturn | Phase 4 | Revenue recognised | DR 1200 / CR 4100 + COGS |
| Demurrage | Phase 5 | DR/CR Demurrage R/I | DR 1250 / CR 4150 |
| Invoice | Phase 6 | AR created | Already in Phase 4 |
| Settlement | Phase 7 | AR/AP cleared | DR/CR 1100 |
| Month-End | Phase 9 | Period locked | All zeros |

---
*Option B — LNG Lifecycle Notebook | Linked to [`lng_feasibility.ipynb`](lng_feasibility.ipynb) and [`lng_feasibility_extended.ipynb`](lng_feasibility_extended.ipynb)*